# Serving resource accounting and actual local timing

Browser Python lab: run in order; cells share variables. Charts come from the code you execute.


## 1 · Account for KV memory

Count both K and V; heads means KV heads. This is an estimate, not a measurement of GPU memory.

In [ ]:
# The blog provides live charts; standalone Python prints chart data.
if 'display_plot' not in globals():
    def display_plot(x, y, title='', xlabel='', ylabel=''):
        print(title, list(zip(x,y)))

layers, kv_heads, head_dim = 32, 8, 128
bytes_per_element, concurrency = 2, 8
def kv_gib(tokens):
    return 2*layers*kv_heads*head_dim*bytes_per_element*concurrency*tokens/2**30
lengths = [512,1024,2048,4096,8192]
for n in lengths:
    print(n, "tokens =>", kv_gib(n), "GiB")
display_plot(lengths, [kv_gib(n) for n in lengths], "KV estimate", "tokens / request", "GiB")

## 2 · Does tensor sharding preserve results?

This splits the input dimension, requiring a sum of partial products. Output-dimension sharding concatenates results instead.

In [ ]:
x = [2,3,5,7]
W = [[1,2],[3,4],[5,6],[7,8]]
full = [sum(x[i]*W[i][j] for i in range(4)) for j in range(2)]
parts = [[sum(x[i]*W[i][j] for i in shard) for j in range(2)] for shard in ([0,1],[2,3])]
merged = [sum(p[j] for p in parts) for j in range(2)]
print("local contributions:",parts)
print("all-reduce sum:",merged, "reference:",full)
assert merged == full

## 3 · ZeRO training states

16 bytes/parameter is this mixed-precision Adam convention: 2 weights, 2 gradients, 4 master weights, 8 moments. Activations are excluded.

In [ ]:
P, ranks = 7_000_000_000, 8
for stage in range(4):
    weights = 2*P/(ranks if stage>=3 else 1)
    gradients = 2*P/(ranks if stage>=2 else 1)
    optimizer = 12*P/(ranks if stage>=1 else 1)
    print("ZeRO",stage,"persistent GiB/rank:",(weights+gradients+optimizer)/2**30)
print("Peak also includes activations, temporary gathered parameters and buffers.")

## 4 · Actual browser CPU timing

This measures Python loops, not CUDA, BLAS or TPU. Fix inputs, warm up, then repeat. Change n to observe growth; do not compare serving frameworks with these numbers.

In [ ]:
import time, statistics
n = 24
matrix = [[(i+j)%7/7 for j in range(n)] for i in range(n)]
def matmul():
    return [[sum(matrix[i][k]*matrix[k][j] for k in range(n)) for j in range(n)] for i in range(n)]
matmul()
times = []
for _ in range(12):
    start = time.perf_counter()
    out = matmul()
    times.append((time.perf_counter()-start)*1000)
print("shape:",(n,n),"checksum:",sum(map(sum,out)))
print("median ms:",statistics.median(times),"max ms:",max(times))
display_plot(list(range(1,13)),times,"Measured Python matmul","repeat","ms")